# Personalized Product Recommendations

The goal of this notebook is to recommend products to a visitor based on their previous interactions.

In [1]:
import pandas as pd
import kagglehub

C:\Users\smako\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
events = pd.read_csv("data/events.csv")
events.head(10)

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN
5,1433224086234,972639,view,22556,NaN
6,1433221923240,810725,view,443030,NaN
7,1433223291897,794181,view,439202,NaN
8,1433220899221,824915,view,428805,NaN
9,1433221204592,339335,view,82389,NaN


In [3]:
events["event"].value_counts()

event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

In [4]:
items_per_visitor = events.groupby("visitorid")["itemid"].nunique()

print(items_per_visitor.describe())
print("Visitors with 2+ items:", (items_per_visitor >= 2).sum())
print("Share:", (items_per_visitor >= 2).mean())

count    1.407580e+06
mean     1.524019e+00
std      7.143724e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.814000e+03
Name: itemid, dtype: float64
Visitors with 2+ items: 288080
Share: 0.20466332286619587


In [5]:
event_summary = (
    events.groupby("event")
    .agg(
        event_count=("itemid", "size"),
        unique_visitors=("visitorid", "nunique"),
        unique_items=("itemid", "nunique"),
    )
    .sort_values("event_count", ascending=False)
)

event_summary

,event_count,unique_visitors,unique_items
event,,,
view,2664312,1404179,234838
addtocart,69332,37722,23903
transaction,22457,11719,12025


In [6]:
event_dates = pd.to_datetime(events["timestamp"], unit="ms", utc=True)

print("Start:", event_dates.min())
print("End:", event_dates.max())

Start: 2015-05-03 03:00:04.384000+00:00
End: 2015-09-18 02:59:47.788000+00:00


In [7]:
cutoff = events["timestamp"].quantile(0.8)

train_events = events[events["timestamp"] <= cutoff].copy()
test_events = events[events["timestamp"] > cutoff].copy()

print("Cutoff:", pd.to_datetime(cutoff, unit="ms", utc=True))
print("Train rows:", len(train_events))
print("Test rows:", len(test_events))
print("Last train event:", pd.to_datetime(train_events["timestamp"].max(), unit="ms", utc=True))
print("First test event:", pd.to_datetime(test_events["timestamp"].min(), unit="ms", utc=True))

Cutoff: 2015-08-18 04:23:31.966000+00:00
Train rows: 2204881
Test rows: 551220
Last train event: 2015-08-18 04:23:31.966000+00:00
First test event: 2015-08-18 04:23:37.919000+00:00


In [8]:
train_visitors = set(train_events["visitorid"].unique())
test_visitors = set(test_events["visitorid"].unique())

known_test_visitors = train_visitors & test_visitors
new_test_visitors = test_visitors - train_visitors

print("Unique test visitors:", len(test_visitors))
print("Test visitors with training history:", len(known_test_visitors))
print("Share with training history:", f"{len(known_test_visitors) / len(test_visitors):.1%}")
print("New visitors in test:", len(new_test_visitors))

Unique test visitors: 308548
Test visitors with training history: 24735
Share with training history: 8.0%
New visitors in test: 283813


In [9]:
warm_test_events = test_events[
    test_events["visitorid"].isin(known_test_visitors)
].copy()

print("Test rows from known visitors:", len(warm_test_events))
print("Share of test rows:", f"{len(warm_test_events) / len(test_events):.1%}")

warm_test_summary = (
    warm_test_events.groupby("event")
    .agg(
        event_count=("itemid", "size"),
        unique_visitors=("visitorid", "nunique"),
        unique_items=("itemid", "nunique"),
    )
    .sort_values("event_count", ascending=False)
)

warm_test_summary

Test rows from known visitors: 77836
Share of test rows: 14.1%


,event_count,unique_visitors,unique_items
event,,,
view,73720,24684,32376
addtocart,2775,865,2111
transaction,1341,353,1131


In [10]:
event_weights = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5,
}

weighted_train_events = train_events.copy()
weighted_train_events["strength"] = (
    weighted_train_events["event"].map(event_weights)
)

user_item_interactions = (
    weighted_train_events
    .groupby(["visitorid", "itemid"], as_index=False)["strength"]
    .max()
)

user_item_interactions.head()

,visitorid,itemid,strength
0,1,72028,1
1,2,216305,1
2,2,259884,1
3,2,325215,1
4,2,342816,1


In [11]:
print(user_item_interactions["strength"].value_counts().sort_index())
print("User-item pairs:", len(user_item_interactions))
print("Visitors:", user_item_interactions["visitorid"].nunique())
print("Items:", user_item_interactions["itemid"].nunique())

strength
1    1662336
3      33906
5      16933
Name: count, dtype: int64
User-item pairs: 1713175
Visitors: 1123767
Items: 212916


In [12]:
from scipy.sparse import csr_matrix

user_codes, user_ids = pd.factorize(user_item_interactions["visitorid"])
item_codes, item_ids = pd.factorize(user_item_interactions["itemid"])

user_item_matrix = csr_matrix(
    (
        user_item_interactions["strength"].astype("float32").to_numpy(),
        (user_codes, item_codes),
    ),
    shape=(len(user_ids), len(item_ids)),
)

density = user_item_matrix.nnz / (
    user_item_matrix.shape[0] * user_item_matrix.shape[1]
)

print("Matrix shape:", user_item_matrix.shape)
print("Stored interactions:", user_item_matrix.nnz)
print("Density:", f"{density:.6%}")

Matrix shape: (1123767, 212916)
Stored interactions: 1713175
Density: 0.000716%


In [13]:
import implicit

als_model = implicit.als.AlternatingLeastSquares(
    factors=32,
    regularization=0.1,
    iterations=10,
)

als_model.fit(user_item_matrix)

C:\Users\smako\AppData\Roaming\Python\Python312\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 22 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 10/10 [01:19<00:00,  7.90s/it]


In [14]:
user_id_to_index = {
    visitor_id: index
    for index, visitor_id in enumerate(user_ids)
}

visitor_id = min(known_test_visitors)
visitor_index = user_id_to_index[visitor_id]

recommended_indices, scores = als_model.recommend(
    visitor_index,
    user_item_matrix[visitor_index],
    N=10,
)

recommended_item_ids = item_ids[recommended_indices]

print("Visitor:", visitor_id)
print("Recommendations:")
print(list(zip(recommended_item_ids.tolist(), scores.tolist())))

Visitor: 155
Recommendations:
[(369447, 0.0041960496455430984), (161623, 0.0038673749659210443), (7943, 0.0032374998554587364), (393028, 0.0029090391471982002), (335975, 0.0024513760581612587), (29196, 0.0023354331497102976), (200425, 0.002327781869098544), (274798, 0.0022327923215925694), (62549, 0.002177047310397029), (334920, 0.0021267179399728775)]


In [15]:
warm_intent_targets = test_events[
    test_events["visitorid"].isin(train_visitors)
    & test_events["event"].isin(["addtocart", "transaction"])
][["visitorid", "itemid", "event"]].drop_duplicates()

known_item_targets = warm_intent_targets[
    warm_intent_targets["itemid"].isin(item_ids)
].copy()

train_pairs = user_item_interactions[
    ["visitorid", "itemid"]
].assign(seen_in_train=True)

checked_targets = known_item_targets.merge(
    train_pairs,
    on=["visitorid", "itemid"],
    how="left",
)

test_targets = checked_targets[
    checked_targets["seen_in_train"].isna()
].drop(columns="seen_in_train")

print("Warm test intent pairs:", len(warm_intent_targets))
print("Targets with items known to the model:", len(known_item_targets))
print("New recommendable targets:", len(test_targets))

test_targets.groupby("event").agg(
    target_pairs=("itemid", "size"),
    visitors=("visitorid", "nunique"),
    items=("itemid", "nunique"),
)

Warm test intent pairs: 3732
Targets with items known to the model: 3488
New recommendable targets: 2967


,target_pairs,visitors,items
event,,,
addtocart,1951,696,1720
transaction,1016,266,933


In [16]:
top_10_by_visitor = {}

evaluation_visitors = test_targets["visitorid"].unique()

for visitor_id in evaluation_visitors:
    user_index = user_id_to_index[visitor_id]

    recommended_indices, _ = als_model.recommend(
        user_index,
        user_item_matrix[user_index],
        N=10,
    )

    top_10_by_visitor[visitor_id] = set(
        item_ids[recommended_indices].tolist()
    )

print("Visitors evaluated:", len(top_10_by_visitor))

Visitors evaluated: 714


In [17]:
for event_type in ["addtocart", "transaction"]:
    event_targets = test_targets[test_targets["event"] == event_type]
    visitor_recalls = []

    for visitor_id, visitor_targets in event_targets.groupby("visitorid"):
        relevant_items = set(visitor_targets["itemid"])
        recommended_items = top_10_by_visitor[visitor_id]

        recall = len(relevant_items & recommended_items) / len(relevant_items)
        visitor_recalls.append(recall)

    mean_recall = sum(visitor_recalls) / len(visitor_recalls)

    print(
        f"{event_type} Recall@10: {mean_recall:.4f} "
        f"({len(visitor_recalls)} visitors)"
    )

addtocart Recall@10: 0.0269 (696 visitors)
transaction Recall@10: 0.0260 (266 visitors)


In [18]:
popular_item_ids = (
    user_item_interactions
    .groupby("itemid")["strength"]
    .sum()
    .sort_values(ascending=False)
    .index.to_numpy()
)

baseline_top_10_by_visitor = {}

for visitor_id in test_targets["visitorid"].unique():
    user_index = user_id_to_index[visitor_id]

    seen_item_indices = user_item_matrix[user_index].indices
    seen_item_ids = set(item_ids[seen_item_indices].tolist())

    top_items = [
        item_id
        for item_id in popular_item_ids
        if item_id not in seen_item_ids
    ][:10]

    baseline_top_10_by_visitor[visitor_id] = set(top_items)

In [19]:
for event_type in ["addtocart", "transaction"]:
    event_targets = test_targets[test_targets["event"] == event_type]
    visitor_recalls = []

    for visitor_id, visitor_targets in event_targets.groupby("visitorid"):
        relevant_items = set(visitor_targets["itemid"])
        recommended_items = baseline_top_10_by_visitor[visitor_id]

        recall = len(relevant_items & recommended_items) / len(relevant_items)
        visitor_recalls.append(recall)

    mean_recall = sum(visitor_recalls) / len(visitor_recalls)

    print(
        f"{event_type} popularity Recall@10: {mean_recall:.4f} "
        f"({len(visitor_recalls)} visitors)"
    )

addtocart popularity Recall@10: 0.0088 (696 visitors)
transaction popularity Recall@10: 0.0171 (266 visitors)


In [20]:
validation_cutoff = train_events["timestamp"].quantile(0.8)

model_train_events = train_events[
    train_events["timestamp"] <= validation_cutoff
].copy()

validation_events = train_events[
    train_events["timestamp"] > validation_cutoff
].copy()

print(
    "Validation cutoff:",
    pd.to_datetime(validation_cutoff, unit="ms", utc=True)
)
print("Model training rows:", len(model_train_events))
print("Validation rows:", len(validation_events))

Validation cutoff: 2015-07-26 06:29:44.456000+00:00
Model training rows: 1763905
Validation rows: 440976


In [21]:
validation_weighted_events = model_train_events.copy()
validation_weighted_events["strength"] = (
    validation_weighted_events["event"].map(event_weights)
)

validation_interactions = (
    validation_weighted_events
    .groupby(["visitorid", "itemid"], as_index=False)["strength"]
    .max()
)

validation_user_codes, validation_user_ids = pd.factorize(
    validation_interactions["visitorid"]
)
validation_item_codes, validation_item_ids = pd.factorize(
    validation_interactions["itemid"]
)

validation_matrix = csr_matrix(
    (
        validation_interactions["strength"].astype("float32").to_numpy(),
        (validation_user_codes, validation_item_codes),
    ),
    shape=(len(validation_user_ids), len(validation_item_ids)),
)

print("Matrix shape:", validation_matrix.shape)
print("Stored interactions:", validation_matrix.nnz)

Matrix shape: (895898, 194205)
Stored interactions: 1367504


In [22]:
validation_als_model = implicit.als.AlternatingLeastSquares(
    factors=128,
    regularization=0.5,
    iterations=10,
)

validation_als_model.fit(validation_matrix)

100%|██████████| 10/10 [01:17<00:00,  7.73s/it]


In [23]:
validation_train_visitors = set(
    validation_interactions["visitorid"].unique()
)

validation_targets = validation_events[
    validation_events["visitorid"].isin(validation_train_visitors)
    & validation_events["event"].isin(["addtocart", "transaction"])
][["visitorid", "itemid", "event"]].drop_duplicates()

known_validation_targets = validation_targets[
    validation_targets["itemid"].isin(validation_item_ids)
].copy()

validation_train_pairs = validation_interactions[
    ["visitorid", "itemid"]
].assign(seen_in_train=True)

checked_validation_targets = known_validation_targets.merge(
    validation_train_pairs,
    on=["visitorid", "itemid"],
    how="left",
)

validation_targets = checked_validation_targets[
    checked_validation_targets["seen_in_train"].isna()
].drop(columns="seen_in_train")

print("Validation targets:", len(validation_targets))

validation_targets.groupby("event").agg(
    target_pairs=("itemid", "size"),
    visitors=("visitorid", "nunique"),
    items=("itemid", "nunique"),
)

Validation targets: 2349


,target_pairs,visitors,items
event,,,
addtocart,1639,633,1455
transaction,710,221,672


In [24]:
validation_user_to_index = {
    visitor_id: index
    for index, visitor_id in enumerate(validation_user_ids)
}

validation_popular_item_ids = (
    validation_interactions
    .groupby("itemid")["strength"]
    .sum()
    .sort_values(ascending=False)
    .index.to_numpy()
)

validation_als_top_10 = {}
validation_popularity_top_10 = {}

for visitor_id in validation_targets["visitorid"].unique():
    user_index = validation_user_to_index[visitor_id]

    # ALS recommendations
    recommended_indices, _ = validation_als_model.recommend(
        user_index,
        validation_matrix[user_index],
        N=10,
    )
    validation_als_top_10[visitor_id] = set(
        validation_item_ids[recommended_indices].tolist()
    )

    seen_indices = validation_matrix[user_index].indices
    seen_items = set(validation_item_ids[seen_indices].tolist())

    popular_top_items = [
        item_id
        for item_id in validation_popular_item_ids
        if item_id not in seen_items
    ][:10]

    validation_popularity_top_10[visitor_id] = set(popular_top_items)

In [25]:
def mean_recall_at_10(targets, recommendations, event_type):
    event_targets = targets[targets["event"] == event_type]
    visitor_recalls = []

    for visitor_id, visitor_targets in event_targets.groupby("visitorid"):
        relevant_items = set(visitor_targets["itemid"])
        recommended_items = recommendations[visitor_id]

        recall = len(relevant_items & recommended_items) / len(relevant_items)
        visitor_recalls.append(recall)

    return sum(visitor_recalls) / len(visitor_recalls)


for event_type in ["addtocart", "transaction"]:
    als_recall = mean_recall_at_10(
        validation_targets, validation_als_top_10, event_type
    )
    popularity_recall = mean_recall_at_10(
        validation_targets, validation_popularity_top_10, event_type
    )

    print(
        f"{event_type}: "
        f"ALS={als_recall:.4f}, "
        f"Popularity={popularity_recall:.4f}"
    )

addtocart: ALS=0.0354, Popularity=0.0214
transaction: ALS=0.0543, Popularity=0.0219


## Final Model Evaluation

The model configuration was selected using the validation set:

- Algorithm: Alternating Least Squares (ALS)
- Factors: 128
- Regularization: 0.5
- Iterations: 10

In [26]:
final_weighted_events = train_events.copy()
final_weighted_events["strength"] = (
    final_weighted_events["event"].map(event_weights)
)

final_interactions = (
    final_weighted_events
    .groupby(["visitorid", "itemid"], as_index=False)["strength"]
    .max()
)

final_user_codes, final_user_ids = pd.factorize(
    final_interactions["visitorid"]
)
final_item_codes, final_item_ids = pd.factorize(
    final_interactions["itemid"]
)

final_user_item_matrix = csr_matrix(
    (
        final_interactions["strength"].astype("float32").to_numpy(),
        (final_user_codes, final_item_codes),
    ),
    shape=(len(final_user_ids), len(final_item_ids)),
)

final_user_to_index = {
    visitor_id: index
    for index, visitor_id in enumerate(final_user_ids)
}

final_als_model = implicit.als.AlternatingLeastSquares(
    factors=128,
    regularization=0.5,
    iterations=10,
)

final_als_model.fit(final_user_item_matrix)

print("Matrix shape:", final_user_item_matrix.shape)
print("Stored interactions:", final_user_item_matrix.nnz)

100%|██████████| 10/10 [00:55<00:00,  5.56s/it]

Matrix shape: (1123767, 212916)
Stored interactions: 1713175


## Result of test

In [27]:
test_visitor_ids = test_targets["visitorid"].unique()

final_als_top_10 = {}

for visitor_id in test_visitor_ids:
    user_index = final_user_to_index[visitor_id]

    item_indices, scores = final_als_model.recommend(
        user_index,
        final_user_item_matrix[user_index],
        N=10,
        filter_already_liked_items=True,
    )

    final_als_top_10[visitor_id] = set(final_item_ids[item_indices])

for event_type in ["addtocart", "transaction"]:
    als_recall = mean_recall_at_10(
        test_targets,
        final_als_top_10,
        event_type,
    )

    popularity_recall = mean_recall_at_10(
        test_targets,
        baseline_top_10_by_visitor,
        event_type,
    )

    print(
        f"{event_type}: "
        f"ALS={als_recall:.4f}, "
        f"Popularity={popularity_recall:.4f}"
    )

addtocart: ALS=0.0228, Popularity=0.0088
transaction: ALS=0.0134, Popularity=0.0171
